# Campagne FPGA TSA

Notebook de campagne FPGA pour tester l'IP HLS avec le patch courant:


**Workflow:**
1. Executer la cellule 1 (config + menu interactif).
2. Configurer (limiter signaux, choisir PRN, definir `RETRY_FILES` en cellule 2 si besoin — meme usage que **Campagne_CPU_TSA_LUT**).
3. Executer la cellule 2 (campagne FPGA avec resume).

Pour re-tester uniquement certains signaux sans relancer toute la campagne: avec `resume_from_partial=True`, remplir `RETRY_FILES` en cellule 2 avec les noms comme dans la colonne `file` du partial (basename; un chemin complet est accepte).


In [ ]:
# ==========================================================
# CELLULE 1 - CONFIGURATION (AUTONOME)
# ==========================================================
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
    HAS_DISPLAY = True
except Exception:
    HAS_DISPLAY = False

try:
    from ipywidgets import Dropdown, IntText, SelectMultiple, VBox, Label, Output
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False

try:
    import sys as _sys
    import os as _os
    _sys.path.insert(0, _os.path.dirname(_os.path.abspath(__file__)) if "__file__" in dir() else ".")
except Exception:
    pass

try:
    from acq_tsa_lut_cpu import acquisition_stsa as _cpu_acquisition_stsa
    HAS_CPU_LUT = True
except Exception as _e:
    HAS_CPU_LUT = False
    print(f"[WARN] acq_tsa_lut_cpu non disponible, fallback CPU désactivé: {_e}")

try:
    from acq_tsa_lut_cpu import acquisition_stsa as _cpu_acquisition_stsa
    HAS_CPU_LUT = True
except Exception as _cpu_e:
    HAS_CPU_LUT = False
    print(f"[WARN] acq_tsa_lut_cpu non disponible, fallback CPU desactive: {_cpu_e}")

try:
    from pynq import Overlay, allocate
    HAS_PYNQ = True
except Exception:
    HAS_PYNQ = False
    Overlay = None
    allocate = None

FS_DEFAULT = 11_999_000.0
IF_DEFAULT = 3_563_000.0
N_DEFAULT = 11999
NB_PHASES_DEFAULT = 1023
EPS = 1e-12

DDS_LUT_SIZE = 1024
DDS_PHASE_BITS = 32
DDS_LUT_BITS = 10

HEADER_RE = re.compile(
    r"PRN=(?P<prn>-?\d+)"
    r".*SNR=(?P<snr>-?\d+(?:\.\d+)?)dB"
    r".*Doppler=(?P<doppler>-?\d+(?:\.\d+)?)Hz"
    r".*CA_SHIFT=(?P<ca>-?\d+)chips",
    re.IGNORECASE,
)

METHOD_LABELS = {
    "cpu_lut_angles": "CPU LUT angles",
    "fpga": "FPGA",
}

EXPORT_DIR_DEFAULT = "Resultats validation TSA"

def _methods_export_tag(cfg):
    methods = [str(m) for m in cfg.get("run_methods", []) if str(m)]
    if not methods:
        return "no_method"
    return "_vs_".join(methods)


def resolve_export_dir(cfg):
    base = Path(str(cfg.get("export_dir", EXPORT_DIR_DEFAULT)))
    if bool(cfg.get("export_methods_subdir", True)):
        return base / _methods_export_tag(cfg)
    return base


DEFAULT_CONFIG = {
    "signals_dir": "signals",
    "prn_dir": "PRN",
    "signal_glob": "*.csv",
    "limit_signals": None,
    "selected_signal_files": None,
    "run_methods": ["fpga"],
    "prn_list": None,
    "fs_hz": FS_DEFAULT,
    "if_hz": IF_DEFAULT,
    "n_samples": N_DEFAULT,
    "nb_phases": NB_PHASES_DEFAULT,
    "fd_start": -10000,
    "fd_end": 10000,
    "fd_step": 500,
    "detection_ratio_min": 2.5,
    "sw_peak_power_ratio_min": 20.0,
    "winner_margin_db_min": 3.0,
    "doppler_tol_hz": 500,
    "phase_tol_chip": 3,
    "bit_path": "tsa.bit",  # bitstream resynth avec patch (24,4 + fallback coarse)
    "ip_name": "acquisition_stsa_top_0",
    "prn_mode": "all_available",
    "prn_search_mode": "injected_plus_decoys",
    "pfa_decoy_count": 8,
    "pfa_decoy_seed": 42,
    "resume_from_partial": True,
    "export_prefix": "validation_fpga_tsa",  # patch types 24,4 + fallback coarse
    "export_dir": "Resultats validation TSA/campagne_fpga_tsa",
    "export_methods_subdir": False,
    "autosave_every_acq": 10,
    "fpga_debug": True,
    "fpga_debug_poll_s": 1.0,
    "fpga_diag_capture": True,
    "fpga_wait_ready_timeout_s": 2.0,
    "fpga_ap_done_timeout_s": 20.0,
    "fpga_dma_wait_timeout_s": 4.0,
    "fpga_max_retries": 3,
    "fpga_corr_out_best_effort": True,
    "cpu_lut_fallback": True,  # relance CPU LUT si FPGA skipe la fine search
}


def collect_signal_files_for_ui(base_dir):
    base_dir = Path(base_dir)
    if not base_dir.exists():
        return []
    files = sorted(base_dir.glob("*.csv"))
    valid_files = []
    for file_path in files:
        try:
            with file_path.open("r", encoding="utf-8", errors="ignore") as handle:
                first = handle.readline().strip()
            if HEADER_RE.search(first):
                valid_files.append(file_path.name)
        except Exception:
            continue
    return valid_files


def get_available_prns_for_ui(base_dir):
    base_dir = Path(base_dir)
    if not base_dir.exists():
        return []
    prns = []
    patterns = ["PRN-*.csv", "prn-*.csv", "PRN_*.csv", "prn_*.csv"]
    for pattern in patterns:
        for file_path in base_dir.glob(pattern):
            match = re.search(r"(\d+)", file_path.stem)
            if match:
                prns.append(int(match.group(1)))
    return sorted(set(prns))


WIDGET_STATE = {
    "enabled": False,
    "signal_mode": None,
    "limit_value": None,
    "signal_selector": None,
    "fd_mode": None,
    "fd_start": None,
    "fd_end": None,
    "fd_step": None,
    "prn_mode": None,
    "prn_selector": None,
    "prn_search_mode": None,
}


def get_current_config():
    cfg = dict(DEFAULT_CONFIG)
    if not WIDGET_STATE["enabled"]:
        return cfg

    signal_mode = int(WIDGET_STATE["signal_mode"].value)
    if signal_mode == 0:
        cfg["limit_signals"] = None
        cfg["selected_signal_files"] = None
    elif signal_mode == 1:
        cfg["limit_signals"] = int(WIDGET_STATE["limit_value"].value)
        cfg["selected_signal_files"] = None
    else:
        cfg["limit_signals"] = None
        cfg["selected_signal_files"] = [str(WIDGET_STATE["signal_selector"].value)] if WIDGET_STATE["signal_selector"].value else None

    if int(WIDGET_STATE["fd_mode"].value) == 0:
        cfg["fd_start"] = -10000
        cfg["fd_end"] = 10000
        cfg["fd_step"] = 500
    else:
        cfg["fd_start"] = int(WIDGET_STATE["fd_start"].value)
        cfg["fd_end"] = int(WIDGET_STATE["fd_end"].value)
        cfg["fd_step"] = int(WIDGET_STATE["fd_step"].value)

    if int(WIDGET_STATE["prn_mode"].value) == 0:
        cfg["prn_list"] = None
    else:
        cfg["prn_list"] = list(WIDGET_STATE["prn_selector"].value)

    psm = int(WIDGET_STATE["prn_search_mode"].value)
    cfg["prn_search_mode"] = "injected_only" if psm == 0 else ("injected_plus_decoys" if psm == 1 else "full_grid")
    return cfg


_available_signals = collect_signal_files_for_ui(DEFAULT_CONFIG["signals_dir"])
_available_prns = get_available_prns_for_ui(DEFAULT_CONFIG["prn_dir"])

print("=== CONFIGURATION INTERACTIVE ===")
print(f"Signaux trouves: {len(_available_signals)}")
print(f"PRN disponibles: {_available_prns}")

if HAS_WIDGETS and HAS_DISPLAY and _available_prns:
    signal_mode = Dropdown(
        options=[("Tous", 0), ("Limiter", 1), ("Fichier precis", 2)],
        value=0,
        description="Signaux",
    )
    limit_value = IntText(value=min(10, max(1, len(_available_signals))), description="Nb")
    signal_selector = Dropdown(
        options=[("-- choisir --", "")] + [(name, name) for name in _available_signals],
        value="",
        description="Fichier",
    )
    fd_mode = Dropdown(options=[("Standard", 0), ("Personnalise", 1)], value=0, description="Doppler")
    fd_start = IntText(value=DEFAULT_CONFIG["fd_start"], description="Fd debut")
    fd_end = IntText(value=DEFAULT_CONFIG["fd_end"], description="Fd fin")
    fd_step = IntText(value=DEFAULT_CONFIG["fd_step"], description="Fd step")
    prn_mode = Dropdown(options=[("Tous PRN", 0), ("Selection", 1)], value=0, description="PRN")
    prn_selector = SelectMultiple(options=[(str(p), p) for p in _available_prns], value=tuple(_available_prns[:1]), rows=min(10, len(_available_prns)), description="Liste")
    prn_search_mode = Dropdown(
        options=[("Injecte seulement", 0), ("Injecte + leurres", 1), ("Grille complete", 2)],
        value=1,
        description="Recherche",
    )
    config_output = Output()

    def _refresh(_=None):
        cfg = get_current_config()
        with config_output:
            config_output.clear_output()
            print("=== CONFIG EFFECTIVE (selon menu) ===")
            print(pd.Series(cfg, dtype="object"))

    def _on_limit_value_change(change):
        if int(signal_mode.value) == 0 and change.get("new") is not None:
            signal_mode.value = 1

    def _on_signal_selector_change(change):
        if change.get("new"):
            signal_mode.value = 2

    def _on_prn_selector_change(change):
        if int(prn_mode.value) == 0 and change.get("new"):
            prn_mode.value = 1

    def _on_fd_custom_change(change):
        if int(fd_mode.value) == 0:
            fd_mode.value = 1

    for w in [signal_mode, limit_value, signal_selector, fd_mode, fd_start, fd_end, fd_step, prn_mode, prn_selector, prn_search_mode]:
        w.observe(_refresh, names="value")

    limit_value.observe(_on_limit_value_change, names="value")
    signal_selector.observe(_on_signal_selector_change, names="value")
    prn_selector.observe(_on_prn_selector_change, names="value")
    fd_start.observe(_on_fd_custom_change, names="value")
    fd_end.observe(_on_fd_custom_change, names="value")
    fd_step.observe(_on_fd_custom_change, names="value")

    WIDGET_STATE.update({
        "enabled": True,
        "signal_mode": signal_mode,
        "limit_value": limit_value,
        "signal_selector": signal_selector,
        "fd_mode": fd_mode,
        "fd_start": fd_start,
        "fd_end": fd_end,
        "fd_step": fd_step,
        "prn_mode": prn_mode,
        "prn_selector": prn_selector,
        "prn_search_mode": prn_search_mode,
    })

    display(VBox([
        Label("Campagne FPGA STSA (these v2 - patch 24,4 + fallback)"),
        signal_mode, limit_value, signal_selector,
        fd_mode, fd_start, fd_end, fd_step,
        prn_mode, prn_selector,
        prn_search_mode,
        config_output,
    ]))
    _refresh()
else:
    print("[INFO] Widgets non disponibles ou PRN introuvables: configuration par defaut utilisee.")

CONFIG = get_current_config()
print()
print("[INFO] La configuration effective se met a jour dans le panneau interactif ci-dessus.")


In [ ]:
# ==========================================================
# CELLULE 2 - CAMPAGNE DE VALIDATION FPGA
# ==========================================================
FPGA_CONTEXT = None

# Offsets AXI-Lite (HLS acquisition_stsa_top)
AP_CTRL = 0x00
DOPPLER_OUT_DATA = 0x10
PHASE_OUT_DATA = 0x20
PEAK_OUT_LO_DATA = 0x30
PEAK_OUT_HI_DATA = 0x34
DETECTED_OUT_DATA = 0x48
MAX_POWER_OUT_DATA = 0x58
MEAN_POWER_OUT_DATA = 0x68

# AXI DMA status bits
DMASR_HALTED = 1 << 0
DMASR_IDLE = 1 << 1
DMASR_IOC_IRQ = 1 << 12
DMASR_ERR_MASK = (1 << 4) | (1 << 5) | (1 << 6) | (1 << 8) | (1 << 9) | (1 << 10)

# AXIS corr_out depth from acq_stsaV2.h:
# AXIS_OUT_DEPTH = NB_DOPPLER_FINE * 192 * 3 + 1 = 12097
STSA_AXIS_OUT_DEPTH = 21 * 192 * 3 + 1


def to_s32(x):
    x = int(x)
    return x if x < 0x80000000 else x - 0x100000000


def read_u48(ip):
    lo = int(ip.read(PEAK_OUT_LO_DATA)) & 0xFFFFFFFF
    hi = int(ip.read(PEAK_OUT_HI_DATA)) & 0xFFFF
    return (hi << 32) | lo


def peak_out_to_float(raw_u48):
    # power_t is ap_ufixed<48,32> => 16 fractional bits.
    return float(raw_u48) / float(1 << 16)


def phase_error_mod(measured, expected, modulo=1023):
    d = (int(measured) - int(expected)) % int(modulo)
    return int(min(d, modulo - d))


def parse_meta_from_header(csv_file):
    csv_file = Path(csv_file)
    with csv_file.open("r", encoding="utf-8", errors="ignore") as handle:
        first = handle.readline().strip()
    match = HEADER_RE.search(first)
    if not match:
        raise ValueError(f"Metadonnees introuvables dans {csv_file.name}")
    return {
        "prn": int(match.group("prn")),
        "snr": float(match.group("snr")),
        "doppler": int(round(float(match.group("doppler")))),
        "ca_shift": int(match.group("ca")),
    }


def load_signal_pm1(csv_file):
    csv_file = Path(csv_file)
    with csv_file.open("r", encoding="utf-8", errors="ignore") as handle:
        line1 = handle.readline().strip()
        line2 = handle.readline().strip()
    if line1 and not line1.startswith("#") and not line2:
        line2 = line1
    if not line2:
        raise ValueError(f"Signal vide: {csv_file}")
    data = np.fromstring(line2, sep=",", dtype=np.float32)
    if data.size == 0:
        raise ValueError(f"Format invalide: {csv_file}")
    data = np.where(data == 255, -1, data).astype(np.float32)
    return data


def load_prn_sequence(prn_dir, prn, n_samples):
    prn_dir = Path(prn_dir)
    candidates = [
        prn_dir / f"PRN-{prn}.csv",
        prn_dir / f"prn-{prn}.csv",
        prn_dir / f"PRN_{prn}.csv",
        prn_dir / f"prn_{prn}.csv",
    ]
    prn_path = next((path for path in candidates if path.exists()), None)
    if prn_path is None:
        raise FileNotFoundError(f"PRN {prn} introuvable dans {prn_dir}")
    seq = np.genfromtxt(prn_path, delimiter=",", dtype=np.float64)
    seq = np.atleast_1d(seq).reshape(-1)
    if seq.size < n_samples:
        raise ValueError(f"PRN trop court ({seq.size}) dans {prn_path}, attendu >= {n_samples}")
    return seq[:n_samples].astype(np.float32)


def get_available_prns(prn_dir):
    prn_dir = Path(prn_dir)
    prns = []
    patterns = ["PRN-*.csv", "prn-*.csv", "PRN_*.csv", "prn_*.csv"]
    for pattern in patterns:
        for file_path in prn_dir.glob(pattern):
            match = re.search(r"(\d+)", file_path.stem)
            if match:
                prns.append(int(match.group(1)))
    return sorted(set(prns))


def collect_signal_files(signals_dir, signal_glob):
    signals_dir = Path(signals_dir)
    files = sorted(signals_dir.glob(signal_glob)) if signals_dir.exists() else []
    if not files and signals_dir.exists():
        files = sorted(signals_dir.glob("*.csv"))
    valid_files = []
    for file_path in files:
        try:
            parse_meta_from_header(file_path)
            valid_files.append(file_path)
        except Exception:
            continue
    return valid_files


def get_fpga_context(cfg):
    global FPGA_CONTEXT
    if FPGA_CONTEXT is not None:
        return FPGA_CONTEXT
    if not HAS_PYNQ:
        raise RuntimeError("pynq indisponible dans ce kernel")
    overlay = Overlay(cfg["bit_path"], download=True)
    FPGA_CONTEXT = {
        "overlay": overlay,
        "ip": getattr(overlay, cfg["ip_name"]),
        "rx_dma": getattr(overlay, "axi_dma_0"),
        "prn_dma": getattr(overlay, "axi_dma_1"),
        "out_dma": getattr(overlay, "axi_dma_2"),
    }
    return FPGA_CONTEXT


def _dma_channel_is_idle(ch):
    try:
        return bool(getattr(ch, "idle"))
    except Exception:
        return True


def _hard_reset_dma(dma):
    """Reset hardware AXI DMA via MMIO (bit 2 du DMACR)."""
    if dma is None:
        return
    try:
        mmio = getattr(dma, "mmio", None) or getattr(dma, "_mmio", None)
        if mmio is None:
            return
        mmio.write(0x00, 0x04)
        mmio.write(0x30, 0x04)
        for _ in range(100):
            cr_tx = mmio.read(0x00) & 0x04
            cr_rx = mmio.read(0x30) & 0x04
            if cr_tx == 0 and cr_rx == 0:
                break
            time.sleep(0.001)
        mmio.write(0x00, 0x01)
        mmio.write(0x30, 0x01)
    except Exception:
        pass


def _recover_dma_and_ip(ctx, cfg=None, reload_overlay=False):
    global FPGA_CONTEXT
    try:
        ctx["ip"].write(0x00, 0x00)
    except Exception:
        pass

    for key in ("rx_dma", "prn_dma", "out_dma"):
        _hard_reset_dma(ctx.get(key))

    time.sleep(0.05)

    if reload_overlay and cfg is not None:
        try:
            overlay = Overlay(cfg["bit_path"], download=True)
            FPGA_CONTEXT = {
                "overlay": overlay,
                "ip": getattr(overlay, cfg["ip_name"]),
                "rx_dma": getattr(overlay, "axi_dma_0"),
                "prn_dma": getattr(overlay, "axi_dma_1"),
                "out_dma": getattr(overlay, "axi_dma_2"),
            }
            ctx.update(FPGA_CONTEXT)
            print("[DMA-RECOVERY] overlay recharge")
            time.sleep(0.1)
        except Exception as reload_exc:
            print(f"[DMA-RECOVERY] echec reload overlay: {reload_exc}")


def _dma_regs_snapshot(ctx):
    try:
        rx_sr = int(ctx["rx_dma"].mmio.read(0x04))
        prn_sr = int(ctx["prn_dma"].mmio.read(0x04))
        out_sr = int(ctx["out_dma"].mmio.read(0x34))
        return f"rx_sr=0x{rx_sr:08X} prn_sr=0x{prn_sr:08X} out_sr=0x{out_sr:08X}"
    except Exception as exc:
        return f"dma_sr_read_error={exc}"


def _dma_regs_snapshot_dict(ctx):
    try:
        return {
            "rx_sr": int(ctx["rx_dma"].mmio.read(0x04)),
            "prn_sr": int(ctx["prn_dma"].mmio.read(0x04)),
            "out_sr": int(ctx["out_dma"].mmio.read(0x34)),
        }
    except Exception:
        return {"rx_sr": -1, "prn_sr": -1, "out_sr": -1}


def _ip_ctrl_snapshot(ip):
    try:
        ctrl = int(ip.read(AP_CTRL))
        ap_start = (ctrl >> 0) & 0x1
        ap_done = (ctrl >> 1) & 0x1
        ap_idle = (ctrl >> 2) & 0x1
        ap_ready = (ctrl >> 3) & 0x1
        return ctrl, ap_start, ap_done, ap_idle, ap_ready
    except Exception:
        return 0, -1, -1, -1, -1


def _wait_ip_ready_before_start(ip, timeout_s=2.0, poll_s=0.001, dbg=False):
    t0 = time.perf_counter()
    t_last = t0
    while (time.perf_counter() - t0) < timeout_s:
        ctrl, _, _, ap_idle, ap_ready = _ip_ctrl_snapshot(ip)
        if ap_idle == 1 or ap_ready == 1:
            return True
        now = time.perf_counter()
        if dbg and (now - t_last) >= 0.5:
            print(
                f"[FPGA-DBG] wait_ready elapsed={now - t0:.3f}s ctrl=0x{ctrl:08X} idle={ap_idle} ready={ap_ready}",
                flush=True,
            )
            t_last = now
        time.sleep(poll_s)
    return False


def _dma_clear_ioc_irq(dma, sr_off):
    try:
        sr = int(dma.mmio.read(sr_off))
        if sr & DMASR_IOC_IRQ:
            dma.mmio.write(sr_off, DMASR_IOC_IRQ)
    except Exception:
        pass


def _dma_wait_done(dma, sr_off, timeout_s=4.0, accept_ioc_only=False):
    t0 = time.perf_counter()
    while (time.perf_counter() - t0) < timeout_s:
        sr = int(dma.mmio.read(sr_off))
        if sr & DMASR_ERR_MASK:
            return False, sr

        idle = (sr & DMASR_IDLE) != 0
        ioc = (sr & DMASR_IOC_IRQ) != 0

        if idle or (accept_ioc_only and ioc):
            return True, sr

        time.sleep(0.001)

    sr = int(dma.mmio.read(sr_off))
    return False, sr


def fpga_acquisition(signal, prn, cfg):
    t_total0 = time.perf_counter()

    diag = {
        "status": "INIT",
        "attempt": 0,
        "ctrl_start": 0,
        "ctrl_done": 0,
        "ctrl_error": 0,
        "rx_sr_start": 0,
        "prn_sr_start": 0,
        "out_sr_start": 0,
        "rx_sr_done": 0,
        "prn_sr_done": 0,
        "out_sr_done": 0,
        "rx_sr_error": 0,
        "prn_sr_error": 0,
        "out_sr_error": 0,
        "error": "",
    }

    ctx = get_fpga_context(cfg)
    ip = ctx["ip"]
    n_samples = int(cfg["n_samples"])
    ratio_min = float(cfg["detection_ratio_min"])
    total_outputs = STSA_AXIS_OUT_DEPTH

    dbg = bool(cfg.get("fpga_debug", True))
    dbg_poll_s = float(cfg.get("fpga_debug_poll_s", 1.0))
    wait_ready_timeout_s = float(cfg.get("fpga_wait_ready_timeout_s", 2.0))
    ap_done_timeout_s = float(cfg.get("fpga_ap_done_timeout_s", 20.0))
    dma_wait_timeout_s = float(cfg.get("fpga_dma_wait_timeout_s", 4.0))
    max_attempts = int(cfg.get("fpga_max_retries", 3))
    corr_out_best_effort = bool(cfg.get("fpga_corr_out_best_effort", True))

    in_sig = allocate(shape=(n_samples,), dtype=np.int32)
    in_prn = allocate(shape=(n_samples,), dtype=np.int32)
    out_buf = allocate(shape=(total_outputs,), dtype=np.int32)

    in_sig[:] = np.rint(np.real(np.asarray(signal[:n_samples]))).astype(np.int32)
    in_prn[:] = np.rint(np.real(np.asarray(prn[:n_samples]))).astype(np.int32)
    out_buf[:] = 0
    prep_done = time.perf_counter()

    try:
        last_exc = None
        for attempt in range(1, max_attempts + 1):
            try:
                diag["attempt"] = attempt
                if dbg:
                    ctrl, _, ap_done0, ap_idle0, ap_ready0 = _ip_ctrl_snapshot(ip)
                    sr0 = _dma_regs_snapshot_dict(ctx)
                    diag["ctrl_start"] = ctrl
                    diag["rx_sr_start"] = sr0["rx_sr"]
                    diag["prn_sr_start"] = sr0["prn_sr"]
                    diag["out_sr_start"] = sr0["out_sr"]
                    print(
                        f"[FPGA-DBG] attempt={attempt}/{max_attempts} n={n_samples} out={total_outputs} ctrl=0x{ctrl:08X} done={ap_done0} idle={ap_idle0} ready={ap_ready0} {_dma_regs_snapshot(ctx)}",
                        flush=True,
                    )

                ready_ok = _wait_ip_ready_before_start(
                    ip,
                    timeout_s=wait_ready_timeout_s,
                    poll_s=0.001,
                    dbg=dbg,
                )
                if not ready_ok:
                    ctrl, _, _, ap_idle, ap_ready = _ip_ctrl_snapshot(ip)
                    raise TimeoutError(
                        f"IP not ready before start: ctrl=0x{ctrl:08X} idle={ap_idle} ready={ap_ready}"
                    )

                t_dma0 = time.perf_counter()
                # Clear stale IOC irq before posting a new RX transfer.
                _dma_clear_ioc_irq(ctx["out_dma"], 0x34)
                if dbg:
                    print("[FPGA-DBG] submit out_dma.recvchannel.transfer", flush=True)
                ctx["out_dma"].recvchannel.transfer(out_buf)

                if dbg:
                    print("[FPGA-DBG] write ap_start then submit RX/PRN DMA", flush=True)
                ip.write(AP_CTRL, 0x01)
                ctx["rx_dma"].sendchannel.transfer(in_sig)
                ctx["prn_dma"].sendchannel.transfer(in_prn)
                t_dma_submit_done = time.perf_counter()

                t_ip_wait0 = time.perf_counter()
                t_last_dbg = t_ip_wait0
                while True:
                    ctrl, _, ap_done, ap_idle, ap_ready = _ip_ctrl_snapshot(ip)
                    if ap_done == 1:
                        if dbg:
                            sr_done = _dma_regs_snapshot_dict(ctx)
                            diag["ctrl_done"] = ctrl
                            diag["rx_sr_done"] = sr_done["rx_sr"]
                            diag["prn_sr_done"] = sr_done["prn_sr"]
                            diag["out_sr_done"] = sr_done["out_sr"]
                            print(
                                f"[FPGA-DBG] AP_DONE ctrl=0x{ctrl:08X} idle={ap_idle} ready={ap_ready} {_dma_regs_snapshot(ctx)}",
                                flush=True,
                            )
                        break

                    now = time.perf_counter()
                    if dbg and (now - t_last_dbg) >= dbg_poll_s:
                        print(
                            f"[FPGA-DBG] waiting ap_done elapsed={now - t_ip_wait0:.3f}s ctrl=0x{ctrl:08X} idle={ap_idle} ready={ap_ready} {_dma_regs_snapshot(ctx)}",
                            flush=True,
                        )
                        t_last_dbg = now

                    if (now - t_ip_wait0) > ap_done_timeout_s:
                        raise TimeoutError(
                            f"Timeout ap_done after {ap_done_timeout_s}s | ctrl=0x{ctrl:08X} idle={ap_idle} ready={ap_ready} {_dma_regs_snapshot(ctx)}"
                        )
                    time.sleep(0.001)
                t_ip_wait1 = time.perf_counter()

                t_dma_wait0 = time.perf_counter()
                if dbg:
                    print("[FPGA-DBG] wait rx_dma done", flush=True)
                ok_rx, sr_rx = _dma_wait_done(ctx["rx_dma"], 0x04, timeout_s=dma_wait_timeout_s, accept_ioc_only=True)
                if not ok_rx:
                    raise TimeoutError(f"DMA RX not done (sr=0x{sr_rx:08X})")

                if dbg:
                    print("[FPGA-DBG] wait prn_dma done", flush=True)
                ok_prn, sr_prn = _dma_wait_done(ctx["prn_dma"], 0x04, timeout_s=dma_wait_timeout_s, accept_ioc_only=True)
                if not ok_prn:
                    raise TimeoutError(f"DMA PRN not done (sr=0x{sr_prn:08X})")

                if dbg:
                    print("[FPGA-DBG] wait out_dma done", flush=True)
                ok_out, sr_out = _dma_wait_done(ctx["out_dma"], 0x34, timeout_s=dma_wait_timeout_s, accept_ioc_only=True)
                corr_out_state = "OK"
                if not ok_out:
                    if corr_out_best_effort:
                        corr_out_state = f"TIMEOUT_SR_0x{sr_out:08X}"
                        print(f"[DMA-WARN] corr_out incomplete, continue with AXI-Lite outputs (sr=0x{sr_out:08X})", flush=True)
                        _recover_dma_and_ip(ctx, cfg=cfg, reload_overlay=False)
                        ip = ctx["ip"]
                    else:
                        raise TimeoutError(f"DMA OUT not done (sr=0x{sr_out:08X})")

                # Ack completion irq after successful receive side completion.
                _dma_clear_ioc_irq(ctx["out_dma"], 0x34)

                t_dma_wait1 = time.perf_counter()

                arr = np.asarray(out_buf, dtype=np.float64)
                peak = float(np.max(arr))
                mean_corr = float(np.mean(arr))
                peak_ratio = peak / mean_corr if mean_corr > EPS else 0.0
                t_total1 = time.perf_counter()

                dma_submit_s = float(t_dma_submit_done - t_dma0)
                ip_wait_s = float(t_ip_wait1 - t_ip_wait0)
                dma_wait_s = float(t_dma_wait1 - t_dma_wait0)
                dma_total_s = dma_submit_s + dma_wait_s
                kernel_s = ip_wait_s

                detected_hw = int(ip.read(DETECTED_OUT_DATA) & 0x1)
                doppler_hw = int(to_s32(ip.read(DOPPLER_OUT_DATA)))
                phase_hw = int(to_s32(ip.read(PHASE_OUT_DATA)))
                peak_hw_raw = read_u48(ip)
                peak_hw = peak_out_to_float(peak_hw_raw)
                max_power_hw = int(to_s32(ip.read(MAX_POWER_OUT_DATA)))
                mean_power_hw = int(to_s32(ip.read(MEAN_POWER_OUT_DATA)))

                _sw_ratio_min = float(cfg.get("sw_peak_power_ratio_min", 0))
                if _sw_ratio_min > 0 and mean_power_hw > 0:
                    _hw_ratio = max_power_hw / mean_power_hw
                    detected_sw = int(detected_hw == 1 and _hw_ratio >= _sw_ratio_min)
                    if dbg and detected_hw != detected_sw:
                        print(f"[SW-FILTER] override detected {detected_hw}->{detected_sw} (hw_ratio={_hw_ratio:.2f} < sw_min={_sw_ratio_min})", flush=True)
                    detected_hw = detected_sw

                # --- CPU LUT fallback ---
                # Si l'IP a skippe la fine search (mean_power_hw==0 et detected==0)
                # on relance l'algo TSA en float64 sur le meme signal CSV.
                # Cela corrige les cas borderline hors-grille DDS a -20dB
                # sans recompiler l'IP HLS.
                if (not detected_hw) and (mean_power_hw == 0) and bool(cfg.get("cpu_lut_fallback", False)) and HAS_CPU_LUT:
                    try:
                        _sig_fb = load_signal_pm1(sig_file)
                        _prn_fb = load_prn_sequence(cfg["prn_dir"], prn_tested, cfg.get("n_samples", N_DEFAULT))
                        _fb = _cpu_acquisition_stsa(_sig_fb, _prn_fb)
                        if _fb.get("detected"):
                            detected_hw = 1
                            doppler_hw = int(_fb["doppler"])
                            phase_hw = int(_fb["phase"])
                            peak_hw = float(_fb["peak"])
                            max_power_hw = int(_fb.get("max_power", 0))
                            mean_power_hw = int(_fb.get("mean_power", 0) * 1e6)  # scale pour uniformite
                            if dbg:
                                print(f"[CPU-FALLBACK] detected via CPU LUT: doppler={doppler_hw} phase={phase_hw} coarse_var={_fb.get('coarse_var',0):.5f}", flush=True)
                        else:
                            if dbg:
                                print(f"[CPU-FALLBACK] CPU LUT aussi non-detecte (coarse_var={_fb.get('coarse_var',0):.5f})", flush=True)
                    except Exception as _fb_e:
                        if dbg:
                            print(f"[CPU-FALLBACK] erreur: {_fb_e}", flush=True)

                if dbg:
                    print(
                        f"[FPGA-DBG] success detected_hw={detected_hw} doppler={doppler_hw} phase={phase_hw} peak_hw={peak_hw:.3f} peak_hw_raw={peak_hw_raw} peak_axis={peak} ratio={peak_ratio:.4f} max_out={max_power_hw} mean_out={mean_power_hw}",
                        flush=True,
                    )
                    print(
                        f"[FPGA-DBG] timing prepare={prep_done - t_total0:.3f}s submit={dma_submit_s:.3f}s ip_wait={ip_wait_s:.3f}s dma_wait={dma_wait_s:.3f}s total={t_total1 - t_total0:.3f}s",
                        flush=True,
                    )

                diag["status"] = "OK"
                return {
                    "method": "fpga",
                    "detected": detected_hw,
                    "detected_ratio": int(peak_ratio >= ratio_min),
                    "doppler": doppler_hw,
                    "phase": phase_hw,
                    "peak": peak_hw,
                    "peak_raw_u48": float(peak_hw_raw),
                    "peak_axis_max": peak,
                    "peak_ratio": peak_ratio,
                    "max_power_out": max_power_hw,
                    "mean_power_out": mean_power_hw,
                    "corr_out_state": corr_out_state,
                    "time_s": kernel_s,
                    "time_kernel_s": kernel_s,
                    "time_prepare_s": float(prep_done - t_total0),
                    "time_dma_submit_s": dma_submit_s,
                    "time_dma_wait_s": dma_wait_s,
                    "time_dma_s": dma_total_s,
                    "time_ip_wait_s": ip_wait_s,
                    "time_end_to_end_s": float(t_total1 - t_total0),
                    "diag": dict(diag),
                }
            except Exception as exc:
                last_exc = exc
                diag["status"] = "ERROR"
                diag["error"] = str(exc)
                ctrl_e, _, _, _, _ = _ip_ctrl_snapshot(ip)
                sr_e = _dma_regs_snapshot_dict(ctx)
                diag["ctrl_error"] = ctrl_e
                diag["rx_sr_error"] = sr_e["rx_sr"]
                diag["prn_sr_error"] = sr_e["prn_sr"]
                diag["out_sr_error"] = sr_e["out_sr"]
                print(f"[DMA-RECOVERY] tentative {attempt}/{max_attempts} apres erreur: {exc}", flush=True)
                print(
                    f"[DIAG] attempt={attempt} ctrl=0x{ctrl_e:08X} rx_sr=0x{sr_e['rx_sr']:08X} prn_sr=0x{sr_e['prn_sr']:08X} out_sr=0x{sr_e['out_sr']:08X}",
                    flush=True,
                )
                if dbg:
                    ctrl, _, ap_done_e, ap_idle_e, ap_ready_e = _ip_ctrl_snapshot(ip)
                    print(
                        f"[FPGA-DBG] before recovery ctrl=0x{ctrl:08X} done={ap_done_e} idle={ap_idle_e} ready={ap_ready_e} {_dma_regs_snapshot(ctx)}",
                        flush=True,
                    )
                reload_needed = attempt >= max_attempts - 1
                _recover_dma_and_ip(ctx, cfg=cfg, reload_overlay=reload_needed)
                if reload_needed:
                    ctx = get_fpga_context(cfg)
                    ip = ctx["ip"]
                if dbg:
                    ctrl, _, ap_done_r, ap_idle_r, ap_ready_r = _ip_ctrl_snapshot(ip)
                    print(
                        f"[FPGA-DBG] after recovery ctrl=0x{ctrl:08X} done={ap_done_r} idle={ap_idle_r} ready={ap_ready_r} {_dma_regs_snapshot(ctx)}",
                        flush=True,
                    )

        err = RuntimeError(f"Echec FPGA apres recovery DMA: {last_exc}")
        setattr(err, "diag", dict(diag))
        raise err
    finally:
        in_sig.freebuffer()
        in_prn.freebuffer()
        out_buf.freebuffer()


def run_method(method_name, signal, prn_seq, cfg):
    if method_name == "fpga":
        return fpga_acquisition(signal, prn_seq, cfg)
    raise ValueError(f"Methode inconnue dans ce notebook: {method_name}")


def build_winner_table(df_raw, cfg):
    winner_rows = []
    for (file_name, method_name), group in df_raw.groupby(["file", "method"], dropna=False):
        valid_group = group[group["status"] == "ok"].sort_values("peak_ratio", ascending=False).reset_index(drop=True)
        if valid_group.empty:
            error_row = group.iloc[0]
            winner_rows.append({
                "file": file_name,
                "method": method_name,
                "status": error_row["status"],
                "error_message": error_row.get("error_message", ""),
            })
            continue

        top1 = valid_group.iloc[0]
        doppler_err = int(abs(int(top1["doppler_meas_hz"]) - int(top1["doppler_true_hz"])))
        phase_err = int(phase_error_mod(int(top1["phase_meas_chip"]), int(top1["phase_true_chip"]), int(cfg["nb_phases"])))
        strict_success = int((int(top1["detected"]) == 1) and (int(top1["prn_tested"]) == int(top1["prn_injected"])) and (doppler_err <= int(cfg["doppler_tol_hz"])) and (phase_err <= int(cfg["phase_tol_chip"])))

        winner_rows.append({
            "file": file_name,
            "method": method_name,
            "status": "ok",
            "error_message": "",
            "snr_db": float(top1["snr_db"]),
            "prn_injected": int(top1["prn_injected"]),
            "prn_winner": int(top1["prn_tested"]),
            "doppler_true_hz": int(top1["doppler_true_hz"]),
            "doppler_meas_hz": int(top1["doppler_meas_hz"]),
            "doppler_err_hz": doppler_err,
            "phase_true_chip": int(top1["phase_true_chip"]),
            "phase_meas_chip": int(top1["phase_meas_chip"]),
            "phase_err_chip": phase_err,
            "peak": float(top1["peak"]),
            "peak_ratio": float(top1["peak_ratio"]),
            "time_s": float(top1["time_s"]),
            "strict_success": strict_success,
        })
    return pd.DataFrame(winner_rows)


cfg = get_current_config()
CONFIG = dict(cfg)
# Notebook en mode FPGA pur: desactive explicitement le fallback CPU LUT.
cfg["cpu_lut_fallback"] = False
CONFIG["cpu_lut_fallback"] = False

signals_dir = Path(cfg["signals_dir"])
prn_dir = Path(cfg["prn_dir"])
signal_files = collect_signal_files(signals_dir, cfg["signal_glob"])
if cfg.get("selected_signal_files"):
    _selected = set(str(x) for x in cfg["selected_signal_files"])
    signal_files = [p for p in signal_files if p.name in _selected]
elif cfg.get("limit_signals"):
    signal_files = signal_files[: int(cfg["limit_signals"])]
if not signal_files:
    raise FileNotFoundError("Aucun signal apres filtrage (verifie le fichier selectionne ou le nombre limite).")

available_prns = get_available_prns(prn_dir)
selected_prns = [int(prn) for prn in cfg["prn_list"] if int(prn) in available_prns] if cfg.get("prn_list") else available_prns

_psm = str(cfg.get("prn_search_mode", "full_grid"))
if _psm == "injected_only":
    _n_prn_per_sig = 1
elif _psm == "injected_plus_decoys":
    _nd = int(cfg.get("pfa_decoy_count", 8))
    _n_prn_per_sig = 1 + min(_nd, max(0, len(selected_prns) - 1))
else:
    _n_prn_per_sig = len(selected_prns)

_n_acq = len(signal_files) * _n_prn_per_sig * len(cfg["run_methods"])
print(f"Acquisitions prevues: {_n_acq}")

export_dir = resolve_export_dir(cfg)
export_dir.mkdir(parents=True, exist_ok=True)
prefix = cfg["export_prefix"]
partial_path = export_dir / f"{prefix}_raw_partial.csv"

# Resume / retry: meme logique que Campagne_CPU_TSA_LUT.ipynb
print(f"[DEBUG RESUME] resume_from_partial={cfg.get('resume_from_partial', False)}")
print(f"[DEBUG RESUME] partial_path={partial_path}")
print(f"[DEBUG RESUME] partial_path.exists()={partial_path.exists()}")
# Fichiers a re-tester et remplacer dans les resultats (sans toucher les autres).
# Noms comme la colonne « file » du partial (basename est pris si tu colles un chemin).
# Laisser [] pour un resume normal.
# Exemples de noms (decommente / adapte depuis ton partial):
#   "gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n20_doppler-n6250Hz_ca-804.csv"
#   "gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n20_doppler-n8750Hz_ca-804.csv"
RETRY_FILES = []
# True: ne filtre PAS signal_files par RETRY_FILES et ne supprime PAS les lignes retry
# du raw_partial; la campagne reprend uniquement les signaux restants.
RUN_REMAINING_ONLY = True

def _normalize_retry_basenames(entries):
    out = []
    for x in entries or []:
        if x is None:
            continue
        s = str(x).strip()
        if not s:
            continue
        out.append(Path(s).name)
    return out


_RETRY_BASENAMES = _normalize_retry_basenames(RETRY_FILES)

# Si RETRY_FILES est rempli, on peut soit relancer seulement cette liste,
# soit reprendre uniquement les restants (RUN_REMAINING_ONLY=True).
if _RETRY_BASENAMES and (not RUN_REMAINING_ONLY):
    _retry_set = set(_RETRY_BASENAMES)
    _before = len(signal_files)
    signal_files = [p for p in signal_files if p.name in _retry_set]
    _missing_retry = sorted(_retry_set - {p.name for p in signal_files})
    print(f"[RETRY FILTER] signal_files={len(signal_files)}/{_before}")
    if _missing_retry:
        print(f"[RETRY WARN] absents du dossier signals: {_missing_retry}")
elif _RETRY_BASENAMES and RUN_REMAINING_ONLY:
    print("[RETRY INFO] RUN_REMAINING_ONLY=True: pas de filtrage direct par RETRY_FILES")
    print("[RETRY INFO] Reprise sur signaux restants selon raw_partial")

# Cherche le partial meme si renomme avec " (1)" / " (2)" par OneDrive/Windows
_partial_candidates = [partial_path]
for _suf in [" (1)", " (2)", " (3)"]:
    _partial_candidates.append(partial_path.parent / (partial_path.stem + _suf + partial_path.suffix))
_partial_to_load = next((p for p in _partial_candidates if p.exists()), None)

_done_keys = set()
new_rows = []
diag_rows = []
if cfg.get("resume_from_partial", False) and _partial_to_load:
    _prev = pd.read_csv(_partial_to_load)
    if _RETRY_BASENAMES and (not RUN_REMAINING_ONLY):
        _known_files = set(_prev["file"].astype(str).unique())
        _missing_retry = sorted(set(_RETRY_BASENAMES) - _known_files)
        if _missing_retry:
            print(f"[RETRY WARN] Fichiers listes mais absents du partial charge: {_missing_retry}")
        _prev_keep = _prev[~_prev["file"].isin(_RETRY_BASENAMES)]
        _n_retry = len(_prev) - len(_prev_keep)
        print(f"[RETRY] {_n_retry} entrees exclues pour re-run: {_RETRY_BASENAMES}")
    else:
        _prev_keep = _prev
    new_rows = _prev_keep.to_dict(orient="records")
    for _, _r in _prev_keep.iterrows():
        _done_keys.add((str(_r["file"]), int(_r["prn_tested"])))
    print(f"[RESUME] {len(new_rows)} lignes chargees depuis {_partial_to_load.name}")
    print(f"[RESUME] Acquisitions restantes: ~{_n_acq - len(_done_keys)} sur {_n_acq}")
    del _prev, _prev_keep
elif cfg.get("resume_from_partial", False) and _RETRY_BASENAMES and (_partial_to_load is None):
    print("[RETRY] raw_partial introuvable: execution sur RETRY_FILES uniquement")
    print("[FRESH] Starting new campaign from scratch")
else:
    print("[FRESH] Starting new campaign from scratch")

prn_cache = {}
_acq_counter = len(new_rows)
for index_signal, sig_file in enumerate(signal_files, 1):
    meta = parse_meta_from_header(sig_file)
    prn_injected = int(meta["prn"])
    doppler_true_hz = int(meta["doppler"])
    phase_true_chip = int(meta["ca_shift"] % int(cfg["nb_phases"]))
    snr_db = float(meta["snr"])

    signal = load_signal_pm1(sig_file)
    n_samples = int(cfg["n_samples"])
    signal = np.pad(signal, (0, max(0, n_samples - signal.size)), mode="edge")[:n_samples].astype(np.complex64)

    _others = [int(p) for p in selected_prns if int(p) != int(prn_injected)]
    if _psm == "injected_only" or not _others:
        prns_this_signal = [prn_injected]
    elif _psm == "injected_plus_decoys":
        _k = min(int(cfg.get("pfa_decoy_count", 8)), len(_others))
        _rng = np.random.default_rng(int(cfg.get("pfa_decoy_seed", 42)) + index_signal)
        _pick = _rng.choice(np.array(_others, dtype=np.int64), size=_k, replace=False)
        prns_this_signal = [prn_injected] + sorted(int(x) for x in _pick.tolist())
    else:
        prns_this_signal = selected_prns

    for prn_tested in prns_this_signal:
        if (sig_file.name, prn_tested) in _done_keys:
            continue

        prn_global_t0 = time.perf_counter()
        prn_acq_count = 0
        if prn_tested not in prn_cache:
            prn_cache[prn_tested] = load_prn_sequence(prn_dir, prn_tested, n_samples)
        prn_seq = prn_cache[prn_tested]

        try:
            t_e2e0 = time.perf_counter()
            result = run_method("fpga", signal, prn_seq, cfg)
            t_e2e1 = time.perf_counter()
            end_to_end_s = float(t_e2e1 - t_e2e0)
            if bool(cfg.get("fpga_diag_capture", True)) and isinstance(result.get("diag", None), dict):
                diag_rows.append({
                    "file": sig_file.name,
                    "prn_tested": int(prn_tested),
                    "status": result["diag"].get("status", ""),
                    "attempt": result["diag"].get("attempt", 0),
                    "ctrl_start": result["diag"].get("ctrl_start", 0),
                    "ctrl_done": result["diag"].get("ctrl_done", 0),
                    "ctrl_error": result["diag"].get("ctrl_error", 0),
                    "rx_sr_start": result["diag"].get("rx_sr_start", 0),
                    "prn_sr_start": result["diag"].get("prn_sr_start", 0),
                    "out_sr_start": result["diag"].get("out_sr_start", 0),
                    "rx_sr_done": result["diag"].get("rx_sr_done", 0),
                    "prn_sr_done": result["diag"].get("prn_sr_done", 0),
                    "out_sr_done": result["diag"].get("out_sr_done", 0),
                    "error": result["diag"].get("error", ""),
                })
            _acq_counter += 1
            prn_acq_count += 1
            d_meas = int(result["doppler"])
            p_meas = int(result["phase"])
            d_err = abs(d_meas - int(doppler_true_hz))
            p_err = phase_error_mod(p_meas, int(phase_true_chip), int(cfg["nb_phases"]))
            print(
                f"[{_acq_counter}/{_n_acq}] fpga PRN={prn_tested} SNR={snr_db:g} "
                f"DONE global={end_to_end_s:.3f}s | "
                f"doppler={d_meas:+d}Hz (true={int(doppler_true_hz):+d}, err={d_err}) | "
                f"phase={p_meas} (true={int(phase_true_chip)}, err={p_err}) | "
                f"corr_out={str(result.get('corr_out_state', 'UNKNOWN'))}"
            )
            new_rows.append({
                "file": sig_file.name,
                "method": "fpga",
                "status": "ok",
                "error_message": "",
                "snr_db": snr_db,
                "prn_injected": prn_injected,
                "prn_tested": int(prn_tested),
                "doppler_true_hz": doppler_true_hz,
                "phase_true_chip": phase_true_chip,
                "detected": int(result["detected"]),
                "doppler_meas_hz": int(result["doppler"]),
                "phase_meas_chip": int(result["phase"]),
                "peak": float(result["peak"]),
                "peak_axis_max": float(result.get("peak_axis_max", np.nan)),
                "peak_ratio": float(result["peak_ratio"]),
                "detected_ratio": int(result.get("detected_ratio", result["detected"])),
                "max_power_out": float(result.get("max_power_out", np.nan)),
                "mean_power_out": float(result.get("mean_power_out", np.nan)),
                "corr_out_state": str(result.get("corr_out_state", "UNKNOWN")),
                "time_s": float(result["time_s"]),
                "time_kernel_s": float(result.get("time_kernel_s", result["time_s"])),
                "time_prepare_s": float(result.get("time_prepare_s", np.nan)),
                "time_dma_submit_s": float(result.get("time_dma_submit_s", np.nan)),
                "time_dma_wait_s": float(result.get("time_dma_wait_s", np.nan)),
                "time_dma_s": float(result.get("time_dma_s", np.nan)),
                "time_ip_wait_s": float(result.get("time_ip_wait_s", np.nan)),
                "time_end_to_end_s": end_to_end_s,
            })
        except Exception as exc:
            _acq_counter += 1
            prn_acq_count += 1
            print(f"[{_acq_counter}/{_n_acq}] fpga PRN={prn_tested} SNR={snr_db:g} ERROR: {exc}")
            diag_err = getattr(exc, "diag", {}) if bool(cfg.get("fpga_diag_capture", True)) else {}
            if isinstance(diag_err, dict) and diag_err:
                diag_rows.append({
                    "file": sig_file.name,
                    "prn_tested": int(prn_tested),
                    "status": diag_err.get("status", "ERROR"),
                    "attempt": diag_err.get("attempt", 0),
                    "ctrl_start": diag_err.get("ctrl_start", 0),
                    "ctrl_done": diag_err.get("ctrl_done", 0),
                    "ctrl_error": diag_err.get("ctrl_error", 0),
                    "rx_sr_start": diag_err.get("rx_sr_start", 0),
                    "prn_sr_start": diag_err.get("prn_sr_start", 0),
                    "out_sr_start": diag_err.get("out_sr_start", 0),
                    "rx_sr_done": diag_err.get("rx_sr_done", 0),
                    "prn_sr_done": diag_err.get("prn_sr_done", 0),
                    "out_sr_done": diag_err.get("out_sr_done", 0),
                    "error": diag_err.get("error", str(exc)),
                })
            new_rows.append({
                "file": sig_file.name,
                "method": "fpga",
                "status": "error",
                "error_message": str(exc),
                "snr_db": snr_db,
                "prn_injected": prn_injected,
                "prn_tested": int(prn_tested),
                "doppler_true_hz": doppler_true_hz,
                "phase_true_chip": phase_true_chip,
                "detected": 0,
                "doppler_meas_hz": np.nan,
                "phase_meas_chip": np.nan,
                "peak": np.nan,
                "peak_axis_max": np.nan,
                "peak_ratio": np.nan,
                "detected_ratio": np.nan,
                "max_power_out": np.nan,
                "mean_power_out": np.nan,
                "corr_out_state": "ERROR",
                "time_s": np.nan,
                "time_kernel_s": np.nan,
                "time_prepare_s": np.nan,
                "time_dma_submit_s": np.nan,
                "time_dma_wait_s": np.nan,
                "time_dma_s": np.nan,
                "time_ip_wait_s": np.nan,
                "time_end_to_end_s": np.nan,
            })

        prn_global_elapsed = time.perf_counter() - prn_global_t0
        print(
            f"[PRN-TOTAL] file={sig_file.name} PRN={prn_tested} acquisitions={prn_acq_count} "
            f"global_elapsed={prn_global_elapsed:.3f}s"
        )

        pd.DataFrame(new_rows).to_csv(partial_path, index=False)

raw_results = pd.DataFrame(new_rows)
winner_results = build_winner_table(raw_results, cfg)

raw_results.to_csv(export_dir / f"{prefix}_raw_results.csv", index=False)
winner_results.to_csv(export_dir / f"{prefix}_winner_results.csv", index=False)
if bool(cfg.get("fpga_diag_capture", True)) and diag_rows:
    pd.DataFrame(diag_rows).to_csv(export_dir / f"{prefix}_dma_diagnostics.csv", index=False)

print("Exports:")
print(export_dir / f"{prefix}_raw_results.csv")
print(export_dir / f"{prefix}_winner_results.csv")
if bool(cfg.get("fpga_diag_capture", True)) and diag_rows:
    print(export_dir / f"{prefix}_dma_diagnostics.csv")
